# OpenTorpedo — Teknofest optimizasyon (Google Colab)

Ağır simülasyon **senin PC'de değil**, Colab sunucusunda çalışır.

**Kısıtlar:** toplam kütle ≤500 g, tel çapı ≤2 mm, bobin çapı ≤16 mm, yay boyu ≤60 mm.

## Kurulum (birini seç)

1. **Zip yükle (en kolay):** Bilgisayarda `python3 scripts/make_colab_zip.py` → `dist/opentorpedo_colab.zip` oluşur. Colab'da sol menüden zip'i yükle, sonra **Hücre 2**.
2. **GitHub (varsayılan):** [efeerdogmus0/opentorpedo](https://github.com/efeerdogmus0/opentorpedo) — Hücre 2 otomatik klonlar.
3. **Google Drive:** Projeyi Drive'a koy; **Hücre 2** içinde `DRIVE_PATH` doldur.

Sonuç: `teknofest_optimized.json` indirilir → PC'deki `configs/` klasörüne kopyala.

In [ ]:
# --- Kaynak seçimi (bir yöntem yeterli) ---
import os
import zipfile
from pathlib import Path

WORKDIR = Path("/content/opentorpedo")
GIT_URL = "https://github.com/efeerdogmus0/opentorpedo.git"
DRIVE_PATH = ""  # Drive kullanacaksan: "/content/drive/MyDrive/opentorpedo"
UPLOAD_ZIP = False  # zip yerine GitHub (varsayılan)

if WORKDIR.exists():
    !rm -rf "{WORKDIR}"
WORKDIR.mkdir(parents=True)
os.chdir(WORKDIR)

if GIT_URL:
    !git clone --depth 1 "{GIT_URL}" .
elif DRIVE_PATH:
    from google.colab import drive
    drive.mount("/content/drive")
    !cp -r "{DRIVE_PATH}"/* .
elif UPLOAD_ZIP:
    from google.colab import files
    print("opentorpedo_colab.zip dosyasını seç…")
    uploaded = files.upload()
    zname = next(n for n in uploaded if n.endswith(".zip"))
    with zipfile.ZipFile(zname, "r") as zf:
        zf.extractall(WORKDIR)
else:
    raise RuntimeError("GIT_URL, DRIVE_PATH veya UPLOAD_ZIP ayarla.")

import sys
sys.path.insert(0, str(WORKDIR))
print("Proje kökü:", WORKDIR)
print("Dosyalar:", sorted(p.name for p in WORKDIR.iterdir())[:12], "…")

In [ ]:
# scipy Colab'da hazır; yine de kontrol
import numpy as np
import scipy
print("numpy", np.__version__, "| scipy", scipy.__version__)

In [ ]:
# --- Optimizasyon ---
# preset: "quick" (~80), "fast" (~160), "medium" (~650), "full" (~3500, önerilen Colab)
from teknofest.grid_search import run_grid_search, count_combos

PRESET = "full"
print(f"Kombinasyon sayısı: {count_combos(PRESET)}")

result = run_grid_search(
    PRESET,
    root=WORKDIR,
    progress_every=100,
)

In [ ]:
# Özet tablo + JSON indir
from IPython.display import display, JSON
from google.colab import files

display(JSON(result))

out = WORKDIR / "configs" / "teknofest_optimized.json"
files.download(str(out))

print("\n=== ÜRETİM ÖZETİ ===")
print(f"Max hız (sim): {result['max_speed_m_s']} m/s")
print(f"Toplam kütle: {result['total_mass_g']} g | CoG: {result['cog_cm']} cm")
b, s = result["ballast"], result["spring"]
print(f"Balast: {b['mass_g']} g @ {b['position_cm']} cm (burundan)")
print(
    f"Yay: tel {s['wire_diameter_mm']} mm, bobin {s['coil_diameter_mm']} mm, "
    f"{s['active_coils']} sarım, L={s['free_length_mm']} mm, sıkışma {s['compression_mm']} mm"
)
print(f"F={s['force_N']} N, Δv≈{s['delta_v_m_s']} m/s")